# Asignación de turnos

## Tipo de problema

Asignar turnos a empleados (en hospitales, call centers, aeropuertos) respetando leyes laborales, preferencias de descansos, bajas y fluctuaciones de carga de trabajo.

## Formas de resolverlo

Los nombres de la primera columna coinciden con la [tabla del README](../README.md#formas-de-resolverlo).

| Enfoque (README) | Aplicación en este problema | Uso en este ejercicio |
|---|---|---|
| Restricciones | Buscar asignaciones mediante reglas de disponibilidad, descanso y compatibilidad. | **Principal** |
| Entera o mixta | Modelar la cobertura y las penalizaciones con variables de asignación. | Alternativa |
| Heurísticas y metaheurísticas | Mejorar cuadrantes grandes con muchas preferencias. | Alternativa |

**Enfoque utilizado:** Restricciones.

**Técnica concreta:** Búsqueda con retroceso y poda por disponibilidad, descansos, carga máxima y coste.

[Asignación de turnos: OR-Tools](https://developers.google.com/optimization/scheduling/employee_scheduling).

## Ejercicio propuesto

Cubrir mañana y tarde de tres días con Ana, Bruno y Carla.  
Cada turno dura 8 horas y requiere una persona. Máximo un turno al día  
por persona y dos turnos en el horizonte (16 h). Quien hace tarde no puede  
hacer la mañana siguiente. Ana no está disponible la tarde del día 2.  
Cada asignación tiene una penalización de preferencia (menor es mejor).  
Objetivo: cubrir todos los turnos minimizando la suma de penalizaciones.  
Método exacto: búsqueda con retroceso y poda por restricciones y coste.  
Las reglas son supuestos del ejercicio, no una representación de legislación.

## Método elegido para este ejercicio

**Enfoque utilizado:** Restricciones.

Búsqueda con retroceso: se asigna cada turno a una persona compatible, respetando disponibilidad, carga máxima y descanso. Se poda cuando el coste parcial no puede mejorar el mejor plan. Las reglas son supuestos didácticos, no legislación laboral.

## Implementación

Ejecuta las celdas de código en orden. Solo se utiliza la biblioteca estándar de Python.

In [1]:
def resolver():
    nombres = ('Ana', 'Bruno', 'Carla')
    # Filas: empleados; columnas: día1-M,T, día2-M,T, día3-M,T.
    penalizaciones = ((0, 3, 0, 9, 0, 3), (2, 0, 2, 0, 2, 0), (1, 1, 1, 1, 1, 1))
    mejor = float('inf'), None
    cargas = [0, 0, 0]

    def buscar(plan, coste):
        nonlocal mejor
        # Todos los costes son no negativos: esta poda conserva el óptimo.
        if coste >= mejor[0]:
            return
        turno = len(plan)
        if turno == 6:
            mejor = coste, tuple(plan)
            return
        for empleado in range(3):
            if cargas[empleado] >= 2 or (turno == 3 and empleado == 0):
                continue
            # La tarde no puede asignarse a quien trabajó esa mañana.
            if turno % 2 == 1 and plan[-1] == empleado:
                continue
            # Una mañana no puede seguir a una tarde del mismo empleado.
            if turno % 2 == 0 and turno > 0 and plan[-1] == empleado:
                continue
            cargas[empleado] += 1
            buscar(plan + [empleado], coste + penalizaciones[empleado][turno])
            cargas[empleado] -= 1

    buscar([], 0)
    coste, plan = mejor
    assert plan is not None and len(plan) == 6
    assert all(plan.count(e) <= 2 for e in range(3))
    assert all(plan[t] != plan[t + 1] for t in range(5))
    assert plan[3] != 0
    return coste, tuple(nombres[e] for e in plan)

## Resolución y resultados

In [2]:
coste, plan = resolver()
for dia in range(3):
    print(f'Día {dia + 1}: mañana={plan[2 * dia]}, tarde={plan[2 * dia + 1]}')
print(f'Penalización mínima: {coste}')

Día 1: mañana=Ana, tarde=Bruno
Día 2: mañana=Carla, tarde=Bruno
Día 3: mañana=Ana, tarde=Carla
Penalización mínima: 2


## Interpretación

Mañana/tarde: día 1 **Ana/Bruno**, día 2 **Carla/Bruno**, día 3 **Ana/Carla**. La penalización mínima es **2**. Cada persona trabaja dos turnos; nadie enlaza una tarde con la mañana siguiente.

**Alcance:** los datos son sintéticos. Las garantías y supuestos se refieren a esta instancia; al aumentar su tamaño, la enumeración puede resultar costosa.